# 2-5. RAG 평가: RAGAS

⏱ **소요시간**: 15분  
<!-- optional --> 수업 시간이 부족하면 생략 가능합니다 (Day2 S5에서 다시 등장).

## 학습 목표
- RAGAS 핵심 지표 **faithfulness · answer_relevancy · context_precision · context_recall**의 의미를 구분한다.
- 이전 노트북의 캐프스톤 베이스라인을 평가해본다.
- LangSmith / LangFuse 같은 실행 로깅 도구의 역할을 개념적으로 이해한다.

## 핵심 키워드
`RAGAS` `faithfulness` `answer_relevancy` `context_precision` `context_recall` `LangSmith` `LangFuse`

> 🔒 RAGAS 지표 계산은 **LLM을 judge로 사용**한다. 폐쇄망에서는 동일한 로컬 LLM(Ollama)을 judge로 지정해 완전 오프라인 으로 평가 가능.

In [ ]:
import sys; sys.path.insert(0, '../..')
from common import get_llm, get_embeddings, provider_badge
print(provider_badge())

## 1. 핵심 지표 4가지

| 지표 | 측정 대상 | 한 줄 설명 |
|---|---|---|
| **faithfulness** | 답변 ↔ 컨텍스트 | 답변의 각 주장이 제공된 컨텍스트에서 뒷받침되는지(환각 여부) |
| **answer_relevancy** | 질문 ↔ 답변 | 답변이 질문에 직접적으로 대응하는지 (동떨어진 답변이면 점수↓) |
| **context_precision** | 검색 결과 순서 | 상위 k개 컨텍스트 중 실제로 답변에 사용된 문서가 앞쪽에 있었는지(순위 품질) |
| **context_recall** | 정답 ↔ 검색 | 정답(ground_truth)에 대해 필요한 정보가 얼마나 검색됐는지 |

**해석 순서**: faithfulness가 낮으면 → LLM이 지어낸 것 / context_recall이 낮으면 → retriever 문제 / answer_relevancy가 낮으면 → 프롬프트가 동떨어진.

## 2. 캐프스톤 베이스라인 로드

In [ ]:
import json
from pathlib import Path

baseline_path = Path('./_capstone_baseline.json')
if not baseline_path.exists():
    raise FileNotFoundError('04_rag_pipeline_lcel.ipynb 를 먼저 실행해주세요.')

baseline = json.loads(baseline_path.read_text(encoding='utf-8'))

# 평가용 ground truth — 수업용으로 강사가 사전 작성해둔 정답
GROUND_TRUTH = [
    '14일 이내에 서면으로 철회할 수 있으나, 이미 서비스가 제공되었거나 영업일 기준 3일 이내 증정이 끝난 경우 제한된다.',
    '이용자는 즉시 고객센터 또는 가장 가까운 영업점에 신고해야 하며, 선의무과실이 없다면 신고 이전 사고도 보상책임이 금융회사에 있다.',
    '수수료 인상 적용일 30일 전에 공지해야 한다.',
    '거래일로부터 1년 이내에 이의신청 가능하고, 금융회사는 접수일로부터 15영업일 이내 결과를 통지한다.',
    '원칙적으로 24시간 이용 가능하나, 시스템 점검·장애·운영상 필요에 따라 일시 제한될 수 있으며 사전 공지된다.',
]

for i, row in enumerate(baseline):
    row['ground_truth'] = GROUND_TRUTH[i]

print(f'평가 대상 샘플 {len(baseline)}개 준비 완료')

## 3. RAGAS 평가 실행

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list([
    {
        'question': r['question'],
        'answer': r['answer'],
        'contexts': r['contexts'],
        'ground_truth': r['ground_truth'],
    }
    for r in baseline
])
print(dataset)

In [ ]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# judge LLM과 embedding을 common.get_llm / get_embeddings 로 주입 → 오픈/오프라인 모두 동작
judge_llm = LangchainLLMWrapper(get_llm(temperature=0))
judge_emb = LangchainEmbeddingsWrapper(get_embeddings())

result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=judge_llm,
    embeddings=judge_emb,
)
print(result)

In [ ]:
# 샘플별 점수 테이블화
import pandas as pd

df = result.to_pandas()
cols = [c for c in ['question', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall'] if c in df.columns]
df[cols]

In [ ]:
# 지표별 평균값 시각화
import matplotlib.pyplot as plt

metric_cols = [c for c in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall'] if c in df.columns]
if metric_cols:
    means = df[metric_cols].mean()
    ax = means.plot(kind='barh', figsize=(7, 3), xlim=(0, 1))
    ax.set_xlabel('score')
    ax.set_title('RAGAS metrics — capstone baseline')
    for i, v in enumerate(means):
        ax.text(v + 0.01, i, f'{v:.2f}', va='center')
    plt.tight_layout()
    plt.show()

## 4. 실행 로깅: LangSmith / LangFuse (실행 없이 개념만)

운영 환경에서는 매 질의의 retriever 입출력, 토큰 수, 레이턴시, 프롬프트 별 버전을 남겨두어야 개선점을 찾을 수 있다.

| 도구 | 폐쇄망 | 특징 |
|---|---|---|
| **LangSmith** (랜체인 공식) | ☁️ SaaS 기본 | 자동 트레이싱, 플러그인 불필요, 데이터셋·에발 대시보드 통합 |
| **LangFuse** | 🔒 셀프호스팅 가능 | Docker Compose로 사내 배포, OpenTelemetry 호환 |

**LangFuse 설치 레퍼런스** (에어 갭 모드 수강생용)
```bash
# docker-compose up -d langfuse postgres
# export LANGFUSE_PUBLIC_KEY=... LANGFUSE_SECRET_KEY=... LANGFUSE_HOST=http://langfuse.internal:3000
# from langfuse.callback import CallbackHandler; handler = CallbackHandler()
# rag_chain.invoke(q, config={"callbacks": [handler]})
```

**LangSmith** (예시 — 외부 통신 가능 프로젝트에서만)
```bash
# export LANGCHAIN_TRACING_V2=true LANGCHAIN_API_KEY=...
# 별도 코드 없이 모든 LCEL 체인이 자동 트레이싱됨
```

## 정리

- RAGAS는 **LLM judge 기반**이므로 judge 모델의 성능이 지표 신뢰도의 상한.
- 4개 지표를 **같이** 보고 하락 원인을 진단해야 한다 (retriever 문제 vs 프롬프트 문제 vs 모델 문제).
- 폐쇄망 운영 관측성은 **LangFuse 셀프호스팅** 또는 OpenTelemetry + 내부 로깅 백엔드가 실용적.

## 과제 (선택)
1. `capstone_baseline`의 1번 질문 답변을 의도적으로 **약간의 거짓정보**(예: 14일 → 20일)로 바꿔 넣어보세요. faithfulness 점수가 어떻게 변하나요?
2. `retriever` `k`를 1 → 5 → 10으로 늘려가며 context_precision / recall이 어떻게 움직이는지 관찰하세요.

## 더 읽어보기
- teddylee777/langchain-kr — [12-RAG / RAG-Evaluation](https://github.com/teddylee777/langchain-kr/tree/main/12-RAG)
- [RAGAS 공식 문서](https://docs.ragas.io/en/stable/)
- [LangFuse (셀프호스팅)](https://langfuse.com/docs/deployment/self-host)
- [LangSmith Tracing](https://docs.smith.langchain.com/observability)